In [ ]:
# !pip install langchain-community
# pip install langchain-ollama
from langchain_community.vectorstores import FAISS

In [ ]:

from langchain_ollama import OllamaEmbeddings


In [ ]:
from langchain_community.document_loaders import PyPDFLoader


In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [ ]:
!pip install PyPDFLoad

ERROR: Could not find a version that satisfies the requirement PyPDFLoad (from versions: none)
ERROR: No matching distribution found for PyPDFLoad


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
from langchain_community.document_loaders import (
    CSVLoader, PyPDFLoader, TextLoader
)

folder_path = '/content/drive/MyDrive/Colab Notebooks'  # change this

all_docs = []

for filename in os.listdir(folder_path):z
    filepath = os.path.join(folder_path, filename)

    loader = None
    if filename.endswith('.csv'):
        loader = CSVLoader(filepath)
    elif filename.endswith('.pdf'):
        loader = PyPDFLoader(filepath)
    elif filename.endswith('.txt'):
        loader = TextLoader(filepath)
    else:
        continue  # skip unsupported files

    if loader:
        try:
            docs = loader.load()
            all_docs.extend(docs)
        except LookupError as e:
            print(f"Skipping file '{filename}' due to encoding error: {e}")
        except Exception as e:
            print(f"Skipping file '{filename}' due to an unexpected error: {e}")

print(f"Total docs loaded: {len(all_docs)}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


ERROR:pypdf._cmap:Advanced encoding /SymbolSetEncoding not implemented yet


Skipping file 'standard-treatment-guidelines.pdf' due to encoding error: unknown encoding: /SymbolSetEncoding
Total docs loaded: 5815


In [ ]:
from langchain_text_splitters  import RecursiveCharacterTextSplitter
text_splitter= RecursiveCharacterTextSplitter(
                                chunk_size=1000,
                                chunk_overlap=200)

document = text_splitter.split_documents(all_docs)

In [ ]:
def cleantext(text):
    return text.encode("utf-8","ignore").decode("utf-8","ignore")

newdocs=[]
for d in document:
    d.page_content= cleantext(d.page_content)
    newdocs.append(d)

In [ ]:
a=len(newdocs)
print(a)

20675


In [ ]:
!pip install sentence-transformers
!pip install langchain-huggingface

from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [ ]:
!pip install faiss-cpu
db= FAISS.from_documents(newdocs, embeddings)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 88.9 MB/s eta 0:00:00


In [ ]:
db= FAISS.from_documents(newdoc, embeddings)


In [ ]:
# Load the FAISS index and the embeddings
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_ollama import Ollama
from langchain.chains import create_retrieval_chain
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate

# Re-initialize embeddings with the same model_name used for creation
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

# Load the FAISS index from the local path
# The 'allow_dangerous_deserialization=True' flag is necessary when loading vectorstores
# that might contain custom classes or complex objects from disk.
loaded_db = FAISS.load_local(
    "/content/drive/MyDrive/faiss_index",
    embeddings,
    allow_dangerous_deserialization=True
)

# Create a retriever from the loaded FAISS index
retriever = loaded_db.as_retriever()

# Initialize an LLM (using Ollama as an example)
# Make sure Ollama is running and the specified model (e.g., 'llama2') is downloaded.
# If you're not using Ollama, replace this with your preferred LLM initialization (e.g., OpenAI, Google Generative AI).
llm = Ollama(model="llama2") # You might need to change the model name or configure Ollama

# Define the prompt template for your RAG chain using the user's preferred format
qa_prompt = ChatPromptTemplate.from_template(
    """
You are an expert research assistant.

Use ONLY the provided context to answer the question.
- Do not use prior knowledge
- If answer is missing, say: "Not found in the document."

Be concise and precise.

Context:
{context}

Question:
{input}

Answer:
"""
)

# Create a chain to combine documents and answer questions
question_answer_chain = create_stuff_documents_chain(llm, qa_prompt)

# Create the full RAG chain that retrieves documents and then answers questions
rag_chain = create_retrieval_chain(retriever, question_answer_chain)

# Example of how to ask a question using the RAG chain
# Replace this with a question relevant to the documents you've loaded.
question = "What are the main causes of diabetes according to the documents?"

print(f"Question: {question}")
response = rag_chain.invoke({"input": question})
print(f"Answer: {response['answer']}")

# You can also inspect the context that was used to generate the answer:
print("\n--- Retrieved Context (first 200 chars of each document) ---")
for doc in response["context"]:
    print(doc.page_content[:200] + "...")


In [ ]:
db.save_local('/content/drive/MyDrive/faiss_index')